### Soal 1

In [20]:
import random 
import math
random.seed(42)

# 1. Input User
n = int(input("Jumlah item (n): "))
kapasitas_bin = int(input("Kapasitas bin (C): "))
kumpulan_item = []
print("Masukkan ukuran setiap item:")
for i in range(n):
    item = int(input(f"Item ke-{i+1}: "))
    kumpulan_item.append(item)

T_init = float(input("Temperatur awal (T_init): "))
alpha = float(input("Cooling rate (alpha): "))
max_iter = int(input("Jumlah iterasi (max_iter): "))

# 3. FFD sebagai solusi awal
def solusi_ffd(kumpulan_item, kapasitas_bin):
    n = len(kumpulan_item)  
    
    solusi_awal = [-1] * n
    
    # List untuk mengetahui berapa berat isi tiap bin
    isi_tiap_bin = []

    # Mengurutkan nomor indeks bedasarkan ukuran barang terbesar - terkecil kumpulan item
    urutan_indeks = sorted(range(n), key=lambda i: kumpulan_item[i], reverse=True)

    # Proses First Fit
    for i in urutan_indeks:
        ukuran_item = kumpulan_item[i] #Mendapatkan isi item
        dimasukkan = False 

        #Mengecek bin yang sudah ada satu persatu
        for j in range(len(isi_tiap_bin)):
            if isi_tiap_bin[j] + ukuran_item <= kapasitas_bin:
                isi_tiap_bin[j] += ukuran_item 
                solusi_awal[i] = j #Memasukkan barang i ke bin ke-j
                dimasukkan = True #Mengubah isi status, bahwa telah dimasukkan ke bin
                break
                
        if not dimasukkan:
            isi_tiap_bin.append(ukuran_item)
            index_baru_bin = len(isi_tiap_bin) - 1 #Mendapatkan index jika belum ada bin (wadah sama sekali)
            solusi_awal[i] = index_baru_bin
            
    return solusi_awal


# 3. Neighborhood
def neighborhood(solusi, kumpulan_item, kapasitas_bin):
    solusi_baru = list(solusi) #Membuat salinan 
    
    berat_bin = {}
    #Menghitung total berat di masing-masing bin
    for i in range(len(solusi_baru)):
        id_bin = solusi_baru[i]
        ukuran = kumpulan_item[i]

        # Mengecek apakah bin sudah ada terdata dalam berat_bin
        if id_bin not in berat_bin:
            berat_bin[id_bin] = 0

        berat_bin[id_bin] += ukuran

    # Melakukan pemilihan acak
    aksi = random.choice(['pindah', 'tukar'])

    if aksi == 'pindah':
        # Memindahkan item acak ke bin lain yang masih muat
        item_acak = random.randint(0, len(kumpulan_item) - 1)
        bin_asal = solusi_baru[item_acak]
        ukuran_item = kumpulan_item[item_acak]

        #Mencari daftar bin yang masih muat untuk menerima
        bin_yang_muat = []
        for id_bin, berat in berat_bin.items(): #Melakukan perulangan sebanyak item pada berat bin
            if id_bin != bin_asal and (berat + ukuran_item <= kapasitas_bin):
                bin_yang_muat.append(id_bin) #Memasukkan id bin ke bin yang masih muat

        # Menyediakan bin baru jika tidak ada yang muat menampung 
        bin_kosong_baru = max(berat_bin.keys()) + 1 if berat_bin else 0
        bin_yang_muat.append(bin_kosong_baru)

        # Memilih bin tujuan secara acak lalu memindah
        bin_tujuan = random.choice(bin_yang_muat)
        solusi_baru[item_acak] = bin_tujuan
    else:
            #wap (Tukar) dua item dari bin yang berbeda
            # Pilih barang pertama secara acak
            item1 = random.randint(0, len(kumpulan_item) - 1)
            bin1 = solusi_baru[item1]
            ukuran1 = kumpulan_item[item1]
            
            # Cari kandidat barang kedua yang kardusnya BERBEDA dari barang pertama
            kandidat_item2 = []
            for i in range(len(solusi_baru)):
                if solusi_baru[i] != bin1:
                    kandidat_item2.append(i)
                    
            # Jika ada kandidatnya, pilih satu secara acak
            if kandidat_item2:
                item2 = random.choice(kandidat_item2)
                bin2 = solusi_baru[item2]
                ukuran2 = kumpulan_item[item2]
                
                # CEK KAPASITAS: Apakah kardusnya bakal jebol kalau mereka bertukar tempat?
                berat_baru_bin1 = berat_bin[bin1] - ukuran1 + ukuran2
                berat_baru_bin2 = berat_bin[bin2] - ukuran2 + ukuran1
                
                if berat_baru_bin1 <= kapasitas_bin and berat_baru_bin2 <= kapasitas_bin:
                    # Kalau aman, tukar posisi mereka!
                    solusi_baru[item1] = bin2
                    solusi_baru[item2] = bin1

    return solusi_baru

# 5. Cost Function (Fungsi Penilai)
def hitung_cost(solusi, kumpulan_item, kapasitas_bin):
    # 1. Hitung total berat di masing-masing kardus
    berat_bin = {}
    for i in range(len(solusi)):
        id_bin = solusi[i]
        ukuran = kumpulan_item[i]
        
        if id_bin not in berat_bin:
            berat_bin[id_bin] = 0
        berat_bin[id_bin] += ukuran
        
    # 2. Hitung jumlah kardus (bin) yang terpakai
    jumlah_bin = len(berat_bin)
    
    # 3. Hitung penalti jika ada kardus yang melebihi batas (overcapacity)
    penalti = 0
    for id_bin, berat in berat_bin.items():
        if berat > kapasitas_bin:
            # Beri hukuman (penalti) yang besar supaya program tahu ini susunan yang jelek
            kelebihan_berat = berat - kapasitas_bin
            penalti += (kelebihan_berat * 100) 
            
    # Nilai akhir yang dikembalikan ke program
    total_cost = jumlah_bin + penalti
    
    return total_cost

# Fungsi untuk output yang diharapkan
def cetak_solusi(solusi, kumpulan_item, kapasitas_bin, judul):
    print(f"\n{judul}")

    item_bin = {}
    for i in range(len(solusi)):
        id_bin = solusi[i]
        ukuran = kumpulan_item[i]

        #Jika bin belum ada dicatatan akan dibuatkan list kosong baru
        if id_bin not in item_bin:
            item_bin[id_bin]= []

        #Memasukkan ukuran barang ke bin
        item_bin[id_bin].append(ukuran)

    #Menghitung total bin yang terpakai
    jumlah_bin = len(item_bin)
    print(f"Jumlah Bin: {jumlah_bin}")

    #Isi tiap bin
    urutan_bin =  sorted(item_bin.keys())
    for nomor_urut, id_bin in enumerate(urutan_bin):
        item_di_bin = item_bin[id_bin]
        total_berat = sum(item_di_bin)
        sisa_kapasitas = kapasitas_bin - total_berat

        print(f"Bin {nomor_urut + 1}: items={item_di_bin}, total={total_berat}, sisa={sisa_kapasitas}")
        
    return jumlah_bin

print("SA - BIN PACKING")
print(f"Jumlah Item: {n}, Kapasitas Bin: {kapasitas_bin}")
print(f"Ukuran Item: {kumpulan_item}")
solusi_awal_ffd = solusi_ffd(kumpulan_item, kapasitas_bin)

jumlah_bin_ffd = cetak_solusi(solusi_awal_ffd, kumpulan_item, kapasitas_bin, "SOLUSI AWAL (FFD)")


solusi_sekarang = list(solusi_awal_ffd) # Mulai dari hasil FFD
solusi_terbaik = list(solusi_awal_ffd)  # Simpan sebagai rekor terbaik sementara
T_sekarang = T_init                     # Atur suhu awal

# 2. Mulai Perulangan (Mencari susunan yang lebih baik)

for iterasi in range(max_iter):
    # Buat satu susunan tetangga baru hasil acakan
    solusi_tetangga = neighborhood(solusi_sekarang, kumpulan_item, kapasitas_bin)
    
    # Hitung skor (cost) keduanya
    cost_sekarang = hitung_cost(solusi_sekarang, kumpulan_item, kapasitas_bin)
    cost_tetangga = hitung_cost(solusi_tetangga, kumpulan_item, kapasitas_bin)
    
    # Hitung selisih skor
    delta = cost_tetangga - cost_sekarang
    
    # Kriteria Penerimaan (Acceptance Criteria)
    if delta < 0:
        # Jika tetangga LEBIH BAGUS (cost lebih kecil), langsung terima!
        solusi_sekarang = solusi_tetangga
    else:
        # Jika tetangga LEBIH JELEK, jangan langsung ditolak.
        # Terima dengan peluang tertentu berdasarkan rumus SA: e^(-delta / T)
        peluang = math.exp(-delta / T_sekarang)
        angka_acak = random.random() # Menghasilkan angka acak dari 0.0 hingga 1.0
        
        if angka_acak < peluang:
            solusi_sekarang = solusi_tetangga
            
    # Update rekor solusi terbaik
    if hitung_cost(solusi_sekarang, kumpulan_item, kapasitas_bin) < hitung_cost(solusi_terbaik, kumpulan_item, kapasitas_bin):
        solusi_terbaik = list(solusi_sekarang)
        
    # Turunkan suhu (Cooling Down)
    T_sekarang = T_sekarang * alpha

# Cetak Solusi SA Terbaik
jumlah_bin_sa = cetak_solusi(solusi_terbaik, kumpulan_item, kapasitas_bin, "SOLUSI SA TERBAIK")

# Cek Validasi (Apakah ada penalti di solusi terbaik?)
cost_terbaik = hitung_cost(solusi_terbaik, kumpulan_item, kapasitas_bin)
# Jika cost sama dengan jumlah bin, artinya penaltinya 0 (Semua muat, tidak ada yang jebol)
status_valid = (cost_terbaik == jumlah_bin_sa) 

print(f"\nStatus: {'VALID (semua bin <= kapasitas)' if status_valid else 'TIDAK VALID (ada bin overcapacity)'}")

# Cek Perbaikan dari FFD
selisih_bin = jumlah_bin_ffd - jumlah_bin_sa
if selisih_bin > 0:
    print(f"Perbaikan dari FFD: ya ({selisih_bin} bin berkurang)")
elif selisih_bin == 0:
    print(f"Perbaikan dari FFD: tidak (0 bin berkurang)")
else:
    print(f"Perbaikan dari FFD: tidak (bertambah {-selisih_bin} bin)")


Masukkan ukuran setiap item:
SA - BIN PACKING
Jumlah Item: 10, Kapasitas Bin: 10
Ukuran Item: [7, 5, 3, 4, 2, 6, 8, 1, 3, 5]

SOLUSI AWAL (FFD)
Jumlah Bin: 5
Bin 1: items=[2, 8], total=10, sisa=0
Bin 2: items=[7, 3], total=10, sisa=0
Bin 3: items=[4, 6], total=10, sisa=0
Bin 4: items=[5, 5], total=10, sisa=0
Bin 5: items=[1, 3], total=4, sisa=6

SOLUSI SA TERBAIK
Jumlah Bin: 5
Bin 1: items=[2, 8], total=10, sisa=0
Bin 2: items=[7, 3], total=10, sisa=0
Bin 3: items=[4, 6], total=10, sisa=0
Bin 4: items=[5, 5], total=10, sisa=0
Bin 5: items=[1, 3], total=4, sisa=6

Status: VALID (semua bin <= kapasitas)
Perbaikan dari FFD: tidak (0 bin berkurang)


### Soal 2

In [ ]:
import random
import math

random.seed(42)

# 1. Menerima input dari user

print("=== INPUT DATA QAP ===")
n = int(input("Jumlah fasilitas/lokasi (n): "))

print("\nMasukkan Flow matrix baris per baris (pisahkan angka dengan spasi):")
flow_matrix = []
for i in range(n):
    baris = list(map(int, input(f"Baris {i+1}: ").split()))
    flow_matrix.append(baris)
    
print("\nMasukkan Distance matrix baris per baris (pisahkan angka dengan spasi):")
distance_matrix = []
for i in range(n):
    baris = list(map(int, input(f"Baris {i+1}: ").split()))
    distance_matrix.append(baris)
    
T_init = float(input("\nTemperatur awal (T_init): "))
alpha = float(input("Cooling rate (alpha): "))
max_iter = int(input("Jumlah iterasi (max_iter): "))

# 3. Cost function: Sum(flow[i][j] * distance[solusi[i]][solusi[j]])
def cost_function(solusi, flow, distance):
    total_cost = 0
    n = len(solusi)
    for i in range(n):
        for j in range(n):
            total_cost += flow[i][j] * distance[solusi[i]][solusi[j]]
    return total_cost

# 4. Neighborhood
def get_neighborhood(solusi):
    tetangga = list(solusi)
    # Pilih dua indeks fasilitas yang berbeda secara acak
    idx1, idx2 = random.sample(range(len(solusi)), 2)
    # Tukar lokasi keduanya
    tetangga[idx1], tetangga[idx2] = tetangga[idx2], tetangga[idx1]
    return tetangga

# Fungsi untuk membantu tampilan cetak
def cetak_matriks(matriks, nama, prefix):
    print(f"\n{nama}:")
    n = len(matriks)
    # Cetak Header
    header = "    " + " ".join([f"{prefix}{i}" for i in range(n)])
    print(header)
    # Cetak Isi
    for i in range(n):
        baris = f"{prefix}{i}  " + " ".join([f"{matriks[i][j]:2}" for j in range(n)])
        print(baris)


# 5. Output
print("\nSA - QUADRATIC ASSIGNMENT PROBLEM")
print(f"Jumlah Fasilitas/Lokasi: {n}")

cetak_matriks(flow_matrix, "Flow Matrix", "F")
cetak_matriks(distance_matrix, "Distance Matrix", "L")

# 2. Representasi Solusi Awal
solusi_awal = list(range(n))
cost_awal = cost_function(solusi_awal, flow_matrix, distance_matrix)

print(f"\nAssignment Awal: Fasilitas i -> Lokasi solusi[i]")
print(f"Cost Awal: {cost_awal}")

# --- PROSES SIMULATED ANNEALING ---
solusi_sekarang = list(solusi_awal)
cost_sekarang = cost_awal

solusi_terbaik = list(solusi_awal)
cost_terbaik = cost_awal

T = T_init

for _ in range(max_iter):
    # Cari Tetangga (Swap 2 lokasi acak)
    solusi_tetangga = get_neighborhood(solusi_sekarang)
    cost_tetangga = cost_function(solusi_tetangga, flow_matrix, distance_matrix)
    
    delta = cost_tetangga - cost_sekarang
    
    # Acceptance Criteria
    if delta < 0:
        solusi_sekarang = solusi_tetangga
        cost_sekarang = cost_tetangga
        
        # Update Rekor Terbaik
        if cost_sekarang < cost_terbaik:
            solusi_terbaik = list(solusi_sekarang)
            cost_terbaik = cost_sekarang
    else:
        # Terima solusi lebih buruk dengan peluang tertentu
        peluang = math.exp(-delta / T)
        if random.random() < peluang:
            solusi_sekarang = solusi_tetangga
            cost_sekarang = cost_tetangga
            
    # Cooling schedule
    T *= alpha

# --- HASIL AKHIR ---
print("\nHASIL AKHIR")
print("Assignment Terbaik:")
for fasilitas, lokasi in enumerate(solusi_terbaik):
    print(f"  Fasilitas {fasilitas} -> Lokasi {lokasi}")
    
print(f"\nCost Terbaik: {cost_terbaik}")

# Hitung Persentase Perbaikan
if cost_awal > 0:
    persentase_perbaikan = ((cost_awal - cost_terbaik) / cost_awal) * 100
else:
    persentase_perbaikan = 0
    
print(f"Perbaikan: {persentase_perbaikan:.2f}%")


SA - QUADRATIC ASSIGNMENT PROBLEM
Jumlah Fasilitas/Lokasi: 4

Flow Matrix:
    F0 F1 F2 F3
F0   0  3  7  2
F1   3  0  4  5
F2   7  4  0  6
F3   2  5  6  0

Distance Matrix:
    L0 L1 L2 L3
L0   0  5  8  3
L1   5  0  2  7
L2   8  2  0  4
L3   3  7  4  0

Assignment Awal: Fasilitas i -> Lokasi solusi[i]
Cost Awal: 288

HASIL AKHIR
Assignment Terbaik:
  Fasilitas 0 -> Lokasi 1
  Fasilitas 1 -> Lokasi 0
  Fasilitas 2 -> Lokasi 2
  Fasilitas 3 -> Lokasi 3

Cost Terbaik: 228
Perbaikan: 20.83%


### Soal 3

In [ ]:
import random
import math

# 6. Set random.seed(42) di awal program
random.seed(42)

# 3. Cost function
def hitung_konflik(solusi):
    konflik = 0
    n = len(solusi)
    for i in range(n):
        for j in range(i + 1, n):
            # Cek konflik diagonal: |baris1 - baris2| == |kolom1 - kolom2|
            if abs(i - j) == abs(solusi[i] - solusi[j]):
                konflik += 1
    return konflik

# 4. Neighborhood
def get_neighborhood(solusi):
    #Memindahkan satu queen ke kolom berbeda di baris yang sama.
    tetangga = list(solusi)
    idx1, idx2 = random.sample(range(len(solusi)), 2)
    tetangga[idx1], tetangga[idx2] = tetangga[idx2], tetangga[idx1]
    return tetangga

# Fungsi bantuan untuk visualisasi papan catur (Q untuk queen, . untuk kosong)
def cetak_papan(solusi):
    n = len(solusi)
    print("Visualisasi Papan:")
    for baris in range(n):
        kolom_queen = solusi[baris]
        # Buat list titik (kosong) sebanyak N
        baris_visual = ['.'] * n
        # Ganti titik menjadi Q pada posisi kolom queen
        baris_visual[kolom_queen] = 'Q'
        # Cetak baris dengan spasi
        print(" " + " ".join(baris_visual))

# 1. INPUT TESTING SESUAI SOAL
N = 8
T_init = 100.0
alpha = 0.995
max_iter = 10000

print(f"SA - N-QUEENS PROBLEM")
print(f"Ukuran Papan: {N}x{N}")

# 2. Representasi solusi awal: List [0, 1, 2, ..., N-1]
# Angka di indeks ke-i merepresentasikan "Queen di baris ke-i berada di kolom solusi[i]"
# Menggunakan permutasi langsung menjamin tidak ada konflik baris & kolom.
solusi_awal = list(range(N))
random.shuffle(solusi_awal) # Acak susunan awalnya
konflik_awal = hitung_konflik(solusi_awal)

print(f"\nKonfigurasi Awal (queen[baris] = kolom):")
print(solusi_awal)
print(f"Konflik Awal: {konflik_awal}")

# PROSES SIMULATED ANNEALING
solusi_sekarang = list(solusi_awal)
cost_sekarang = konflik_awal

solusi_terbaik = list(solusi_awal)
cost_terbaik = konflik_awal

T = T_init
iterasi_ditemukan = max_iter

for i in range(max_iter):
    # Jika solusi sempurna (0 konflik) ditemukan, hentikan perulangan (Early Stop)
    if cost_terbaik == 0:
        iterasi_ditemukan = i
        break
        
    solusi_tetangga = get_neighborhood(solusi_sekarang)
    cost_tetangga = hitung_konflik(solusi_tetangga)
    
    delta = cost_tetangga - cost_sekarang
    
    # Acceptance Criteria
    if delta < 0:
        solusi_sekarang = solusi_tetangga
        cost_sekarang = cost_tetangga
        
        # Update Rekor
        if cost_sekarang < cost_terbaik:
            solusi_terbaik = list(solusi_sekarang)
            cost_terbaik = cost_sekarang
    else:
        # Terima solusi lebih buruk (banyak konflik) dengan peluang tertentu
        peluang = math.exp(-delta / T)
        if random.random() < peluang:
            solusi_sekarang = solusi_tetangga
            cost_sekarang = cost_tetangga
            
    # Cooling Schedule
    T *= alpha

# 5. OUTPUT HASIL AKHIR
print("\nHASIL AKHIR")
print("Konfigurasi Terbaik:")
print(solusi_terbaik)
print(f"Konflik: {cost_terbaik}")

status = "SOLUSI DITEMUKAN" if cost_terbaik == 0 else "TIDAK DITEMUKAN"
print(f"Status: {status}")
print(f"Iterasi: {iterasi_ditemukan}")
print("") # Baris kosong

# Visualisasi Papan Akhir
cetak_papan(solusi_terbaik)

SA - N-QUEENS PROBLEM
Ukuran Papan: 8x8

Konfigurasi Awal (queen[baris] = kolom):
[3, 4, 6, 7, 2, 5, 0, 1]
Konflik Awal: 6

HASIL AKHIR
Konfigurasi Terbaik:
[3, 1, 7, 5, 0, 2, 4, 6]
Konflik: 0
Status: SOLUSI DITEMUKAN
Iterasi: 352

Visualisasi Papan:
 . . . Q . . . .
 . Q . . . . . .
 . . . . . . . Q
 . . . . . Q . .
 Q . . . . . . .
 . . Q . . . . .
 . . . . Q . . .
 . . . . . . Q .
